In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm

# Define the BERT-based classifier
class BertClassifier(nn.Module):
    def __init__(self, dropout=0.1):
        super(BertClassifier, self).__init__()
        self.bert = BertForSequenceClassification.from_pretrained(
            'bert-base-uncased',
            num_labels=1,
            problem_type="binary_classification"
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        return outputs.logits

# Function for inference
def predict(model, tokenizer, texts, max_len=128, batch_size=32):
    """
    Perform inference using the trained model.

    Args:
    - model: Trained BERT model.
    - tokenizer: BERT tokenizer.
    - texts: List of input texts for prediction.
    - max_len: Maximum token length (default: 128).
    - batch_size: Batch size for predictions (default: 32).

    Returns:
    - predictions: Binary predictions (0 or 1) for each input text.
    - probabilities: Probabilities corresponding to predictions.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()

    # Tokenize texts
    encoded = tokenizer(
        text=texts,
        add_special_tokens=True,
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )
    input_ids = encoded['input_ids']
    attention_mask = encoded['attention_mask']

    # Create DataLoader for the inputs
    dataset = TensorDataset(input_ids, attention_mask)
    dataloader = DataLoader(dataset, batch_size=batch_size)

    predictions = []
    probabilities = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting"):
            inputs, masks = [b.to(device) for b in batch]
            logits = model(inputs, masks).squeeze()
            probs = torch.sigmoid(logits)

            preds = (probs > 0.5).int()
            try:
                predictions.extend(preds.cpu().numpy())
                probabilities.extend(probs.cpu().numpy())
            except Exception as e:
                print(e, "preds shape:", preds.shape)

    return predictions, probabilities

# Load the trained model
model = BertClassifier()
model.load_state_dict(torch.load('best_bert_model2.pth'))
model.eval()

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3474891/678360974.py:80: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We

In [2]:
import numpy as np
import pandas as pd

In [ ]:
models = {
    'Finetuned GPT-2': [f"../results/filtered_finetuned_gpt2_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD Imitation+DPO': [f"../results/filtered_gpt2_acl_dpo_seed_{i}.csv" for i in range(5)],
    'GPT-2_qwen KD Imitation+DPO': [f"../results/filtered_gpt2_qwen_acl_dpo_seed_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD_llama Imitation+DPO': [f"../results/filtered_gpt2_llama_acl_dpo_seed_seed_{i}.csv" for i in range(5)],
    'Unfiltred_GPT-2_qwen KD Imitation+DPO': [f"../results/unfiltred_gpt2_qwen_acl_dpo_seed_seed_{i}.csv" for i in range(5)],
    'Unfiltred_GPT-2_llama KD Imitation+DPO': [f"../results/unfiltred_gpt2_llama_acl_dpo_seed_seed_{i}.csv" for i in range(5)],
    
    'Mistral Zero-Shot': [f"../results/filtered_zero_shot_mistral_seed_{i}.csv" for i in range(5)],
    'LLama Zero-Shot': [f"../results/unfiltred_llama_zero_shot_seed_{i}.csv" for i in range(5)],
    'Qwen Zero-Shot': [f"../results/unfiltred_qwen_zero_shot_seed_{i}.csv" for i in range(5)],
    'Unfiltred_llama Zero-Shot': [f"../results/unfiltred_llama_zero_shot_seed_{i}.csv" for i in range(5)],
    'Unfiltred_qwen Zero-Shot': [f"../results/unfiltred_qwen_zero_shot_seed_{i}.csv" for i in range(5)],
    
    'Mistral Finetuned': [f"../results/filtered_finetuned_mistral_seed_{i}.csv" for i in range(5)],
    'Qwen Finetuned': [f"../results/filtered_finetuned_qwen_seed_{i}.csv" for i in range(5)],
    'llama Finetuned': [f"../results/filtered_finetuned_llama_seed_{i}.csv" for i in range(5)],
    'Unfilted Qwen Finetuned': [f"../results/unfiltred_finetuned_qwen_seed_{i}.csv" for i in range(5)],
    'Unfiltred llama Finetuned': [f"../results/unfiltred_finetuned_llama_seed_{i}.csv" for i in range(5)],
    
    'GPT-2 KD (Finetuned Mistral)': [f"../results/filtered_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Finetuned qwen)': [f"../results/filtered_gpt2_qwen_KD_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Finetuned llama)': [f"../results/filtered_gpt2_llama_KD_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (UnfiltredFinetuned llama)': [f"../results/unfiltred_gpt2_finetuned_llama_kd_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (UnfiltredFinetuned qwen)': [f"../results/unfiltred_gpt2_finetuned_qwen_kd_seed_{i}.csv" for i in range(5)],
    
    'GPT-2 Mistral (Frozen)': [f"../results/filtered_frozen_mistral_seed_{i}.csv" for i in range(5)],
    'GPT-2 llama (Frozen)': [f"../results/filtered_gpt2_frozen_llama_seed_{i}.csv" for i in range(5)],
    'GPT-2 qwen (Frozen)': [f"../results/filtered_gpt2_frozen_qwen_seed_{i}.csv" for i in range(5)],
    'Unfiltred GPT-2 llama (Frozen)': [f"../results/unfiltred_gpt2_llama_seed_{i}.csv" for i in range(5)],
    'Unfiltred GPT-2 qwen (Frozen)': [f"../results/unfiltered_gpt2_frozen_qwen_seed_{i}.csv" for i in range(5)],
    
    'GPT-2 KD (Imitation)': [f"../results/filtered_gpt2_imitation_seed_{i}.csv" for i in range(5)],
    'GPT-2_qwen_KD (Imitation)': [f"../results/filtered_gpt2_imitation_phase_qwen_seed_{i}.csv" for i in range(5)],
    'GPT-2_llama_KD (Imitation)': [f"../results/filtered_gpt2_imitation_phase_llama_seed_{i}.csv" for i in range(5)],
    'Unfiltred_GPT-2_qwen_KD (Imitation)': [f"../results/unfiltred_gpt2_imitation_phase_qwen_seed_{i}.csv" for i in range(5)],
    'Unfiltred_GPT-2_llama_KD (Imitation)': [f"../results/unfiltred_gpt2_imitation_phase_llama_seed_{i}.csv" for i in range(5)],
}

# Dictionary to store results for all models
all_results = {}

# Process each model
for model_name, file_names in models.items():
    print(f"\n{'='*60}")
    print(f"Processing: {model_name}")
    print(f"{'='*60}")
    
    # List to store percentage of humorous jokes for each file
    percentages = []
    
    for file_name in file_names:
        try:
            # Load the CSV file
            df = pd.read_csv(file_name)
            
            # Get the jokes from the file and clean the data
            # Convert to string, fill NaN with empty string, and filter out empty strings
            jokes = df['generated_punchline'].fillna('').astype(str).tolist()
            
            # Filter out empty strings and very short strings (likely invalid)
            jokes = [joke for joke in jokes if joke.strip() and len(joke.strip()) > 0]
            
            if len(jokes) == 0:
                print(f"Warning: No valid jokes found in {file_name}")
                continue
            
            # Make predictions
            predictions, probabilities = predict(model, tokenizer, jokes)
            
            # Count the number of humorous jokes
            n_humorous = predictions.count(1)
            
            # Calculate percentage for the current file
            percentage_humorous = n_humorous / len(predictions) * 100
            percentages.append(percentage_humorous)
            
            # Print results for the current file
            print(f"File: {file_name}")
            print(f"Total valid jokes: {len(jokes)}")
            print(f"Number of humorous jokes: {n_humorous}")
            print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")
        
        except FileNotFoundError:
            print(f"Warning: File not found - {file_name}")
            continue
        except KeyError as e:
            print(f"Error: Column not found in {file_name}: {str(e)}")
            continue
        except Exception as e:
            print(f"Error processing {file_name}: {str(e)}")
            # Print first few rows to debug
            try:
                df = pd.read_csv(file_name)
                print(f"Columns in file: {df.columns.tolist()}")
                print(f"First few rows of 'generated_punchline':")
                print(df['generated_punchline'].head())
                print(f"Data types: {df['generated_punchline'].dtype}")
            except:
                pass
            continue
    
    # Calculate mean and standard deviation of percentages if we have data
    if percentages:
        mean_percentage = np.mean(percentages)
        std_dev_percentage = np.std(percentages)
        
        # Store results
        all_results[model_name] = {
            'mean': mean_percentage,
            'std': std_dev_percentage,
            'percentages': percentages
        }
        
        # Print overall statistics for this model
        print(f"Statistics for {model_name}:")
        print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
        print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")
    else:
        print(f"No valid data found for {model_name}")

# Print summary table
print(f"\n{'='*80}")
print("SUMMARY OF ALL MODELS")
print(f"{'='*80}")
print(f"{'Model Name':<45} {'Mean %':<15} {'Std Dev %':<15}")
print(f"{'-'*80}")
for model_name, results in all_results.items():
    print(f"{model_name:<45} {results['mean']:>10.2f}% {results['std']:>15.2f}%")

In [ ]:
models = {
    'Finetuned GPT-2': [f"../results/filtered_finetuned_gpt2_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD Imitation+DPO': [f"../results/filtered_gpt2_acl_dpo_seed_{i}.csv" for i in range(5)],
    'GPT-2_qwen KD Imitation+DPO': [f"../results/filtered_gpt2_qwen_acl_dpo_seed_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD_llama Imitation+DPO': [f"../results/filtered_gpt2_llama_acl_dpo_seed_seed_{i}.csv" for i in range(5)],
 
    
    'Mistral Zero-Shot': [f"../results/filtered_zero_shot_mistral_seed_{i}.csv" for i in range(5)],
    'LLama Zero-Shot': [f"../results/unfiltred_llama_zero_shot_seed_{i}.csv" for i in range(5)],
    'Qwen Zero-Shot': [f"../results/unfiltred_qwen_zero_shot_seed_{i}.csv" for i in range(5)],
 
    
    'Mistral Finetuned': [f"../results/filtered_finetuned_mistral_seed_{i}.csv" for i in range(5)],
    'Qwen Finetuned': [f"../results/filtered_finetuned_qwen_seed_{i}.csv" for i in range(5)],
    'llama Finetuned': [f"../results/filtered_finetuned_llama_seed_{i}.csv" for i in range(5)],
    'Unfilted Qwen Finetuned': [f"../results/unfiltred_finetuned_qwen_seed_{i}.csv" for i in range(5)],
    'Unfiltred llama Finetuned': [f"../results/unfiltred_finetuned_llama_seed_{i}.csv" for i in range(5)],
    
    'GPT-2 KD (Finetuned Mistral)': [f"../results/filtered_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Finetuned qwen)': [f"../results/filtered_gpt2_qwen_KD_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Finetuned llama)': [f"../results/filtered_gpt2_llama_KD_seed_{i}.csv" for i in range(5)],
 
    
    'GPT-2 Mistral (Frozen)': [f"../results/filtered_frozen_mistral_seed_{i}.csv" for i in range(5)],
    'GPT-2 llama (Frozen)': [f"../results/filtered_gpt2_frozen_llama_seed_{i}.csv" for i in range(5)],
    'GPT-2 qwen (Frozen)': [f"../results/filtered_gpt2_frozen_qwen_seed_{i}.csv" for i in range(5)],
    'Unfiltred GPT-2 llama (Frozen)': [f"../results/unfiltred_gpt2_llama_seed_{i}.csv" for i in range(5)],
    'Unfiltred GPT-2 qwen (Frozen)': [f"../results/unfiltered_gpt2_frozen_qwen_seed_{i}.csv" for i in range(5)],
    
    'GPT-2 KD (Imitation)': [f"../results/filtered_gpt2_imitation_seed_{i}.csv" for i in range(5)],
    'GPT-2_qwen_KD (Imitation)': [f"../results/filtered_gpt2_imitation_phase_qwen_seed_{i}.csv" for i in range(5)],
    'GPT-2_llama_KD (Imitation)': [f"../results/filtered_gpt2_imitation_phase_llama_seed_{i}.csv" for i in range(5)],

    'filtered GPT-2 llama (LORA)': [f"../results/filtered_gpt2_llama_seed_{i}.csv" for i in range(5)],
    'filtered GPT-2 qwen (LORA)': [f"../results/filtered_gpt2_qwen_seed_{i}.csv" for i in range(5)],
}

# Dictionary to store results for all models
all_results = {}

# Process each model
for model_name, file_names in models.items():
    print(f"\n{'='*60}")
    print(f"Processing: {model_name}")
    print(f"{'='*60}")
    
    # List to store percentage of humorous jokes for each file
    percentages = []
    
    for file_name in file_names:
        try:
            # Load the CSV file
            df = pd.read_csv(file_name)
            
            # Get the jokes from the file and clean the data
            # Convert to string, fill NaN with empty string, and filter out empty strings
            jokes = df['generated_punchline'].fillna('').astype(str).tolist()
            
            # Filter out empty strings and very short strings (likely invalid)
            jokes = [joke for joke in jokes if joke.strip() and len(joke.strip()) > 0]
            
            if len(jokes) == 0:
                print(f"Warning: No valid jokes found in {file_name}")
                continue
            
            # Make predictions
            predictions, probabilities = predict(model, tokenizer, jokes)
            
            # Count the number of humorous jokes
            n_humorous = predictions.count(1)
            
            # Calculate percentage for the current file
            percentage_humorous = n_humorous / len(predictions) * 100
            percentages.append(percentage_humorous)
            
            # Print results for the current file
            print(f"File: {file_name}")
            print(f"Total valid jokes: {len(jokes)}")
            print(f"Number of humorous jokes: {n_humorous}")
            print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")
        
        except FileNotFoundError:
            print(f"Warning: File not found - {file_name}")
            continue
        except KeyError as e:
            print(f"Error: Column not found in {file_name}: {str(e)}")
            continue
        except Exception as e:
            print(f"Error processing {file_name}: {str(e)}")
            # Print first few rows to debug
            try:
                df = pd.read_csv(file_name)
                print(f"Columns in file: {df.columns.tolist()}")
                print(f"First few rows of 'generated_punchline':")
                print(df['generated_punchline'].head())
                print(f"Data types: {df['generated_punchline'].dtype}")
            except:
                pass
            continue
    
    # Calculate mean and standard deviation of percentages if we have data
    if percentages:
        mean_percentage = np.mean(percentages)
        std_dev_percentage = np.std(percentages)
        
        # Store results
        all_results[model_name] = {
            'mean': mean_percentage,
            'std': std_dev_percentage,
            'percentages': percentages
        }
        
        # Print overall statistics for this model
        print(f"Statistics for {model_name}:")
        print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
        print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")
    else:
        print(f"No valid data found for {model_name}")

# Print summary table
print(f"\n{'='*80}")
print("SUMMARY OF ALL MODELS")
print(f"{'='*80}")
print(f"{'Model Name':<45} {'Mean %':<15} {'Std Dev %':<15}")
print(f"{'-'*80}")
for model_name, results in all_results.items():
    print(f"{model_name:<45} {results['mean']:>10.2f}% {results['std']:>15.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/gpt2_obj2_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# Finetuned GPT-2
file_names = [f"../results/filtered_finetuned_gpt2_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# GPT-2 KD Imitation+DPO
file_names = [f"../results/filtered_gpt2_acl_dpo_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# Mistral Zero-Shot
file_names = [f"../results/filtered_zero_shot_mistral_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# GPT-2 KD (Finetuned Mistral)
file_names = [f"../results/filtered_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# GPT-2 Mistral (LoRA)
file_names = [f"../results/filtered_gpt2_mistral_lora_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# GPT-2 Mistral (Frozen)
file_names = [f"../results/filtered_frozen_mistral_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# GPT-2 KD (Imitation)
file_names = [f"../results/filtered_gpt2_imitation_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# GPT-2 Qwen 
file_names = [f"../results/filtered_gpt2_qwen_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# GPT-2 Qwen(Frozen)
file_names = [f"../results/filtered_gpt2_frozen_qwen_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# Mistral Finetuned
file_names = [f"../results/filtered_finetuned_mistral_seed_{i}.csv" for i in range(5)]
percentages = []

for file_name in file_names:
    gpt2_obj2_preds = pd.read_csv(file_name)

    jokes = (
        gpt2_obj2_preds['generated_punchline']
        .dropna()
        .astype(str)
        .tolist()
    )

    predictions, probabilities = predict(model, tokenizer, jokes)

    n_humorous = predictions.count(1)
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)

    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")


In [ ]:
# GPT-2 Qwen 
file_names = [f"../results/filtered_gpt2_qwen_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/gpt2_oneword_2ndvariant_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/gpt2_obj2_2ndvariant_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/gpt2_mistral_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/bart_obj2_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    gpt2_obj2_preds["generated_punchline"] = gpt2_obj2_preds["setup"] + " " + gpt2_obj2_preds["generated_punchline"]
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/finetuned_bart_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    gpt2_obj2_preds["generated_punchline"] = gpt2_obj2_preds["setup"] + " " + gpt2_obj2_preds["generated_punchline"]
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/mistral_gpt2_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/mistral_zero_shot_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/bart_mistral_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    gpt2_obj2_preds["generated_punchline"] = gpt2_obj2_preds["setup"] + " " + gpt2_obj2_preds["generated_punchline"]
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/mistral_finetuned_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    gpt2_obj2_preds["generated_punchline"] = gpt2_obj2_preds["setup"] + " " + gpt2_obj2_preds["generated_punchline"]
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/gpt2_frozen_mistral_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/bart_frozen_mistral_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    gpt2_obj2_preds["generated_punchline"] = gpt2_obj2_preds["setup"] + " " + gpt2_obj2_preds["generated_punchline"]
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/gpt2_mistral_ACL_with_dpo_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/gpt2_mistral_kd(imitation_phase)_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
# List of file names
file_names = [f"../results/gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(3)]

# List to store percentage of humorous jokes for each file
percentages = []

for file_name in file_names:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    jokes = gpt2_obj2_preds['generated_punchline'].tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
file_name = "../acl_paper_and_kd/mistral_instruct_v01_punchlines_train_20k_output.csv"

temp = ["generated_punchline_1", "generated_punchline_2", "generated_punchline_3"]

# List to store percentage of humorous jokes for each file
percentages = []

for tt in temp:
    # Load the CSV file
    gpt2_obj2_preds = pd.read_csv(file_name)
    
    # Get the jokes from the file
    gpt2_obj2_preds["generated_punchline"] = gpt2_obj2_preds["setup"] + " " + gpt2_obj2_preds[tt]
    jokes = gpt2_obj2_preds['generated_punchline'].fillna('').astype(str).tolist()
    
    # Make predictions
    predictions, probabilities = predict(model, tokenizer, jokes)
    
    # Count the number of humorous jokes
    n_humorous = predictions.count(1)
    
    # Calculate percentage for the current file
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    
    # Print results for the current file
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

# Calculate mean and standard deviation of percentages
mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)

# Print overall statistics
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.nn import TransformerEncoder, TransformerEncoderLayer

# Load tokenizer and model
model_id = "microsoft/Phi-3-mini-4k-instruct"
phi_tok = AutoTokenizer.from_pretrained(model_id)
phi_tok.pad_token = phi_tok.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define Custom Transformer Encoder
class CustomTransformerEncoder(nn.Module):
    def __init__(self, input_dim, model_dim, nhead, num_encoder_layers, dim_feedforward, dropout=0.1):
        super(CustomTransformerEncoder, self).__init__()
        self.embedding = nn.Linear(input_dim, model_dim) if input_dim != model_dim else nn.Identity()
        encoder_layer = TransformerEncoderLayer(d_model=model_dim, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.output_layer = nn.Linear(model_dim, input_dim) if input_dim != model_dim else nn.Identity()

    def forward(self, src):
        src = src.to(torch.float32)
        src = self.embedding(src)
        output = self.transformer_encoder(src)
        output = self.output_layer(output)
        return output

# Load base Phi model
phi_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="cuda:0")
phi_model.eval()

# Define Humor Classifier
class HumorBERTModel(nn.Module):
    def __init__(self):
        super(HumorBERTModel, self).__init__()
        self.bert = phi_model
        self.encoder_phi = CustomTransformerEncoder(3072, 512, 8, 6, 2048, 0.1).to(device)
        self.fc1 = nn.Linear(3072, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        self.dropout1 = nn.Dropout(0.4)
        self.dropout2 = nn.Dropout(0.3)
        self.batchnorm1 = nn.BatchNorm1d(3072)
        self.batchnorm2 = nn.BatchNorm1d(512)
        self.batchnorm3 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU(0.1)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            last_hidden_state = self.bert(input_ids, attention_mask=attention_mask, output_hidden_states=True).hidden_states[-1]
        weights = attention_mask * torch.arange(1, last_hidden_state.shape[1] + 1).unsqueeze(0).to(input_ids.device)
        sum_embeddings = torch.sum(last_hidden_state * weights.unsqueeze(-1), dim=1)
        num_tokens = torch.sum(weights, dim=-1).unsqueeze(-1)
        sentence_embeddings = sum_embeddings / num_tokens
        sentence_embeddings = sentence_embeddings.to(torch.float32)
        self.encoder_phi.train()
        word_embeddings = self.encoder_phi(sentence_embeddings)
        norm = self.batchnorm1(word_embeddings)
        drop = self.dropout1(norm)
        hidden1 = self.leaky_relu(self.fc1(drop))
        hidden1_norm = self.batchnorm2(hidden1)
        hidden2 = self.leaky_relu(self.fc2(hidden1_norm))
        hidden2_norm = self.batchnorm3(hidden2)
        hidden3 = self.relu(self.fc3(hidden2_norm))
        hidden3_drop = self.dropout2(hidden3)
        hidden4 = self.relu(self.fc4(hidden3_drop))
        output = self.fc5(hidden4).squeeze(-1)
        return output

# Define Humor Dataset
class HumorDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = torch.tensor(self.labels[idx]).float()
        encoding = phi_tok(text, return_tensors='pt', padding=True, truncation=True)
        return {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "label": label
        }

def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    labels = [item['label'] for item in batch]
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True)
    attention_mask_padded = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True)
    labels = torch.stack(labels)
    return {
        "input_ids": input_ids_padded,
        "attention_mask": attention_mask_padded,
        "label": labels
    }

# Load trained model
model = HumorBERTModel().to(device)
model.load_state_dict(torch.load("../dataset_humcd/386053/humor_classifier.pt", map_location=device))
model.eval()

# Prediction function
def predict_humor_phi(model, tokenizer, texts, batch_size=8):
    model.eval()
    dataset = HumorDataset(texts, [0]*len(texts))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    preds, probs = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Classifying"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask)
            prob = torch.sigmoid(outputs)
            pred = (prob > 0.5).int()
            preds.extend(pred.cpu().numpy())
            probs.extend(prob.cpu().numpy())
    return preds, probs

# Evaluate files
file_names = [f"../results/gpt2_obj2_seed_{i}.csv" for i in range(3)]
percentages = []

for file_name in file_names:
    df = pd.read_csv(file_name)
    jokes = df['generated_punchline'].tolist()
    predictions, probabilities = predict_humor_phi(model, phi_tok, jokes)
    n_humorous = list(predictions).count(1)
    percentage_humorous = n_humorous / len(predictions) * 100
    percentages.append(percentage_humorous)
    print(f"File: {file_name}")
    print(f"Number of humorous jokes: {n_humorous}")
    print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")

mean_percentage = np.mean(percentages)
std_dev_percentage = np.std(percentages)
print("Overall Statistics:")
print(f"Mean percentage of humorous jokes: {mean_percentage:.2f}%")
print(f"Standard deviation of percentages: {std_dev_percentage:.2f}%")

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.nn import TransformerEncoder, TransformerEncoderLayer

# Load tokenizer and model
model_id = "microsoft/Phi-3-mini-4k-instruct"
phi_tok = AutoTokenizer.from_pretrained(model_id)
phi_tok.pad_token = phi_tok.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define Custom Transformer Encoder
class CustomTransformerEncoder(nn.Module):
    def __init__(self, input_dim, model_dim, nhead, num_encoder_layers, dim_feedforward, dropout=0.1):
        super(CustomTransformerEncoder, self).__init__()
        self.embedding = nn.Linear(input_dim, model_dim) if input_dim != model_dim else nn.Identity()
        encoder_layer = TransformerEncoderLayer(d_model=model_dim, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.output_layer = nn.Linear(model_dim, input_dim) if input_dim != model_dim else nn.Identity()

    def forward(self, src):
        src = src.to(torch.float32)
        src = self.embedding(src)
        output = self.transformer_encoder(src)
        output = self.output_layer(output)
        return output

# Load base Phi model
phi_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="cuda:0")
phi_model.eval()

# Define Humor Classifier
class HumorBERTModel(nn.Module):
    def __init__(self):
        super(HumorBERTModel, self).__init__()
        self.bert = phi_model
        self.encoder_phi = CustomTransformerEncoder(3072, 512, 8, 6, 2048, 0.1).to(device)
        self.fc1 = nn.Linear(3072, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        self.dropout1 = nn.Dropout(0.4)
        self.dropout2 = nn.Dropout(0.3)
        self.batchnorm1 = nn.BatchNorm1d(3072)
        self.batchnorm2 = nn.BatchNorm1d(512)
        self.batchnorm3 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU(0.1)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            last_hidden_state = self.bert(input_ids, attention_mask=attention_mask, output_hidden_states=True).hidden_states[-1]
        weights = attention_mask * torch.arange(1, last_hidden_state.shape[1] + 1).unsqueeze(0).to(input_ids.device)
        sum_embeddings = torch.sum(last_hidden_state * weights.unsqueeze(-1), dim=1)
        num_tokens = torch.sum(weights, dim=-1).unsqueeze(-1)
        sentence_embeddings = sum_embeddings / num_tokens
        sentence_embeddings = sentence_embeddings.to(torch.float32)
        self.encoder_phi.train()
        word_embeddings = self.encoder_phi(sentence_embeddings)
        norm = self.batchnorm1(word_embeddings)
        drop = self.dropout1(norm)
        hidden1 = self.leaky_relu(self.fc1(drop))
        hidden1_norm = self.batchnorm2(hidden1)
        hidden2 = self.leaky_relu(self.fc2(hidden1_norm))
        hidden2_norm = self.batchnorm3(hidden2)
        hidden3 = self.relu(self.fc3(hidden2_norm))
        hidden3_drop = self.dropout2(hidden3)
        hidden4 = self.relu(self.fc4(hidden3_drop))
        output = self.fc5(hidden4).squeeze(-1)
        return output

# Define Humor Dataset
class HumorDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = torch.tensor(self.labels[idx]).float()
        encoding = phi_tok(text, return_tensors='pt', padding=True, truncation=True)
        return {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "label": label
        }

def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    labels = [item['label'] for item in batch]
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True)
    attention_mask_padded = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True)
    labels = torch.stack(labels)
    return {
        "input_ids": input_ids_padded,
        "attention_mask": attention_mask_padded,
        "label": labels
    }

# Load trained model
model = HumorBERTModel().to(device)
model.load_state_dict(torch.load("/home/2017025/rpinni01/humour/TST/dataset_humcd/386053/humor_classifier.pt", map_location=device))
model.eval()

# Prediction function
def predict_humor_phi(model, tokenizer, texts, batch_size=8):
    model.eval()
    dataset = HumorDataset(texts, [0]*len(texts))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    preds, probs = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Classifying"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask)
            prob = torch.sigmoid(outputs)
            pred = (prob > 0.5).int()
            preds.extend(pred.cpu().numpy())
            probs.extend(prob.cpu().numpy())
    return preds, probs
# Evaluate multiple models
models = {
    'Finetuned GPT-2': [f"../results/finetuned_gpt2_seed_{i}.csv" for i in range(3)],
    'GPT-2 Mistral ACL+DPO': [f"../results/gpt2_mistral_ACL_with_dpo_seed_{i}.csv" for i in range(3)],
    'Mistral Zero-Shot': [f"../results/mistral_zero_shot_seed_{i}.csv" for i in range(3)],
    'Mistral Finetuned': [f"../results/mistral_finetuned_seed_{i}.csv" for i in range(3)],
    'GPT-2 KD (Finetuned Mistral)': [f"../results/gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(3)],
    'GPT-2 Mistral (LoRA)': [f"../results/gpt2_mistral_seed_{i}.csv" for i in range(3)],
    'GPT-2 Mistral (Frozen)': [f"../results/gpt2_frozen_mistral_seed_{i}.csv" for i in range(3)],
    'GPT-2 KD (Imitation)': [f"../results/gpt2_mistral_kd(imitation_phase)_seed_{i}.csv" for i in range(3)],
    'GPT-2 Obj2': [f"../results/gpt2_obj2_seed_{i}.csv" for i in range(3)],
}

for model_name, file_names in models.items():
    percentages = []
    print(f"\nEvaluating model: {model_name}")
    for file_name in file_names:
        df = pd.read_csv(file_name)
        jokes = df['generated_punchline'].tolist()
        predictions, probabilities = predict_humor_phi(model, phi_tok, jokes)
        n_humorous = list(predictions).count(1)
        percentage_humorous = n_humorous / len(predictions) * 100
        percentages.append(percentage_humorous)
        print(f"File: {file_name}")
        print(f"Number of humorous jokes: {n_humorous}")
        print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")
    mean_percentage = np.mean(percentages)
    std_dev_percentage = np.std(percentages)
    print(f"{model_name} → Mean: {mean_percentage:.2f}%, Std: {std_dev_percentage:.2f}%")


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.nn import TransformerEncoder, TransformerEncoderLayer

# Load tokenizer and model
model_id = "microsoft/Phi-3-mini-4k-instruct"
phi_tok = AutoTokenizer.from_pretrained(model_id)
phi_tok.pad_token = phi_tok.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define Custom Transformer Encoder
class CustomTransformerEncoder(nn.Module):
    def __init__(self, input_dim, model_dim, nhead, num_encoder_layers, dim_feedforward, dropout=0.1):
        super(CustomTransformerEncoder, self).__init__()
        self.embedding = nn.Linear(input_dim, model_dim) if input_dim != model_dim else nn.Identity()
        encoder_layer = TransformerEncoderLayer(d_model=model_dim, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.output_layer = nn.Linear(model_dim, input_dim) if input_dim != model_dim else nn.Identity()

    def forward(self, src):
        src = src.to(torch.float32)
        src = self.embedding(src)
        output = self.transformer_encoder(src)
        output = self.output_layer(output)
        return output

# Load base Phi model
phi_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="cuda:0")
phi_model.eval()

# Define Humor Classifier
class HumorBERTModel(nn.Module):
    def __init__(self):
        super(HumorBERTModel, self).__init__()
        self.bert = phi_model
        self.encoder_phi = CustomTransformerEncoder(3072, 512, 8, 6, 2048, 0.1).to(device)
        self.fc1 = nn.Linear(3072, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        self.dropout1 = nn.Dropout(0.4)
        self.dropout2 = nn.Dropout(0.3)
        self.batchnorm1 = nn.BatchNorm1d(3072)
        self.batchnorm2 = nn.BatchNorm1d(512)
        self.batchnorm3 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU(0.1)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            last_hidden_state = self.bert(input_ids, attention_mask=attention_mask, output_hidden_states=True).hidden_states[-1]
        weights = attention_mask * torch.arange(1, last_hidden_state.shape[1] + 1).unsqueeze(0).to(input_ids.device)
        sum_embeddings = torch.sum(last_hidden_state * weights.unsqueeze(-1), dim=1)
        num_tokens = torch.sum(weights, dim=-1).unsqueeze(-1)
        sentence_embeddings = sum_embeddings / num_tokens
        sentence_embeddings = sentence_embeddings.to(torch.float32)
        self.encoder_phi.train()
        word_embeddings = self.encoder_phi(sentence_embeddings)
        norm = self.batchnorm1(word_embeddings)
        drop = self.dropout1(norm)
        hidden1 = self.leaky_relu(self.fc1(drop))
        hidden1_norm = self.batchnorm2(hidden1)
        hidden2 = self.leaky_relu(self.fc2(hidden1_norm))
        hidden2_norm = self.batchnorm3(hidden2)
        hidden3 = self.relu(self.fc3(hidden2_norm))
        hidden3_drop = self.dropout2(hidden3)
        hidden4 = self.relu(self.fc4(hidden3_drop))
        output = self.fc5(hidden4).squeeze(-1)
        return output

# Define Humor Dataset
class HumorDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = torch.tensor(self.labels[idx]).float()
        encoding = phi_tok(text, return_tensors='pt', padding=True, truncation=True)
        return {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "label": label
        }

def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    labels = [item['label'] for item in batch]
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True)
    attention_mask_padded = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True)
    labels = torch.stack(labels)
    return {
        "input_ids": input_ids_padded,
        "attention_mask": attention_mask_padded,
        "label": labels
    }

# Load trained model
model = HumorBERTModel().to(device)
model.load_state_dict(torch.load("/home/2017025/rpinni01/humour/TST/dataset_humcd/386053/humor_classifier.pt", map_location=device))
model.eval()

# Prediction function
def predict_humor_phi(model, tokenizer, texts, batch_size=8):
    model.eval()
    dataset = HumorDataset(texts, [0]*len(texts))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    preds, probs = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Classifying"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask)
            prob = torch.sigmoid(outputs)
            pred = (prob > 0.5).int()
            preds.extend(pred.cpu().numpy())
            probs.extend(prob.cpu().numpy())
    return preds, probs
# Evaluate multiple models
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.nn import TransformerEncoder, TransformerEncoderLayer

# Load tokenizer and model
model_id = "microsoft/Phi-3-mini-4k-instruct"
phi_tok = AutoTokenizer.from_pretrained(model_id)
phi_tok.pad_token = phi_tok.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define Custom Transformer Encoder
class CustomTransformerEncoder(nn.Module):
    def __init__(self, input_dim, model_dim, nhead, num_encoder_layers, dim_feedforward, dropout=0.1):
        super(CustomTransformerEncoder, self).__init__()
        self.embedding = nn.Linear(input_dim, model_dim) if input_dim != model_dim else nn.Identity()
        encoder_layer = TransformerEncoderLayer(d_model=model_dim, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.output_layer = nn.Linear(model_dim, input_dim) if input_dim != model_dim else nn.Identity()

    def forward(self, src):
        src = src.to(torch.float32)
        src = self.embedding(src)
        output = self.transformer_encoder(src)
        output = self.output_layer(output)
        return output

# Load base Phi model
phi_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="cuda:0")
phi_model.eval()

# Define Humor Classifier
class HumorBERTModel(nn.Module):
    def __init__(self):
        super(HumorBERTModel, self).__init__()
        self.bert = phi_model
        self.encoder_phi = CustomTransformerEncoder(3072, 512, 8, 6, 2048, 0.1).to(device)
        self.fc1 = nn.Linear(3072, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        self.dropout1 = nn.Dropout(0.4)
        self.dropout2 = nn.Dropout(0.3)
        self.batchnorm1 = nn.BatchNorm1d(3072)
        self.batchnorm2 = nn.BatchNorm1d(512)
        self.batchnorm3 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU(0.1)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            last_hidden_state = self.bert(input_ids, attention_mask=attention_mask, output_hidden_states=True).hidden_states[-1]
        weights = attention_mask * torch.arange(1, last_hidden_state.shape[1] + 1).unsqueeze(0).to(input_ids.device)
        sum_embeddings = torch.sum(last_hidden_state * weights.unsqueeze(-1), dim=1)
        num_tokens = torch.sum(weights, dim=-1).unsqueeze(-1)
        sentence_embeddings = sum_embeddings / num_tokens
        sentence_embeddings = sentence_embeddings.to(torch.float32)
        self.encoder_phi.train()
        word_embeddings = self.encoder_phi(sentence_embeddings)
        norm = self.batchnorm1(word_embeddings)
        drop = self.dropout1(norm)
        hidden1 = self.leaky_relu(self.fc1(drop))
        hidden1_norm = self.batchnorm2(hidden1)
        hidden2 = self.leaky_relu(self.fc2(hidden1_norm))
        hidden2_norm = self.batchnorm3(hidden2)
        hidden3 = self.relu(self.fc3(hidden2_norm))
        hidden3_drop = self.dropout2(hidden3)
        hidden4 = self.relu(self.fc4(hidden3_drop))
        output = self.fc5(hidden4).squeeze(-1)
        return output

# Define Humor Dataset
class HumorDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = torch.tensor(self.labels[idx]).float()
        encoding = phi_tok(text, return_tensors='pt', padding=True, truncation=True)
        return {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "label": label
        }

def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    labels = [item['label'] for item in batch]
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True)
    attention_mask_padded = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True)
    labels = torch.stack(labels)
    return {
        "input_ids": input_ids_padded,
        "attention_mask": attention_mask_padded,
        "label": labels
    }

# Load trained model
model = HumorBERTModel().to(device)
model.load_state_dict(torch.load("/home/2017025/rpinni01/humour/TST/dataset_humcd/386053/humor_classifier.pt", map_location=device))
model.eval()

# Prediction function
def predict_humor_phi(model, tokenizer, texts, batch_size=8):
    model.eval()
    dataset = HumorDataset(texts, [0]*len(texts))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    preds, probs = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Classifying"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask)
            prob = torch.sigmoid(outputs)
            pred = (prob > 0.5).int()
            preds.extend(pred.cpu().numpy())
            probs.extend(prob.cpu().numpy())
    return preds, probs
# Evaluate multiple models
models = {
    'Finetuned GPT-2': [f"../results/filtered_finetuned_gpt2_seed_{i}.csv" for i in range(3)],
    'GPT-2 Mistral ACL+DPO': [f"../results/gpt2_mistral_ACL_with_dpo_seed_{i}.csv" for i in range(3)],
    'Mistral Zero-Shot': [f"../results/filtered_mistral_zero_shot_seed_{i}.csv" for i in range(3)],
    'Mistral Finetuned': [f"../results/filtred_mistral_finetuned_seed_{i}.csv" for i in range(3)],
    'GPT-2 KD (Finetuned Mistral)': [f"../results/filtred_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(3)],
    'GPT-2 Mistral (LoRA)': [f"../results/filtered_gpt2_mistral_seed_{i}.csv" for i in range(3)],
    'GPT-2 Mistral (Frozen)': [f"../results/gpt2_frozen_mistral_seed_{i}.csv" for i in range(3)],
    'GPT-2 KD (Imitation)': [f"../results/filtered_gpt2_mistral_kd(imitation_phase)_seed_{i}.csv" for i in range(3)],
    'GPT-2 Obj2': [f"../results/gpt2_obj2_seed_{i}.csv" for i in range(3)],
    'GPT-2 Qwen ': [f"../results/filtered_gpt2_qwen_seed_{i}.csv" for i in range(3)],
    'GPT-2 Qwen(Frozen)': [f"../results/filtered_gpt2_frozen_qwen_seed_{i}.csv" for i in range(3)],
}

for model_name, file_names in models.items():
    percentages = []
    print(f"\nEvaluating model: {model_name}")
    for file_name in file_names:
        df = pd.read_csv(file_name)
        jokes = df['generated_punchline'].tolist()
        predictions, probabilities = predict_humor_phi(model, phi_tok, jokes)
        n_humorous = list(predictions).count(1)
        percentage_humorous = n_humorous / len(predictions) * 100
        percentages.append(percentage_humorous)
        print(f"File: {file_name}")
        print(f"Number of humorous jokes: {n_humorous}")
        print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")
    mean_percentage = np.mean(percentages)
    std_dev_percentage = np.std(percentages)
    print(f"{model_name} → Mean: {mean_percentage:.2f}%, Std: {std_dev_percentage:.2f}%")

for model_name, file_names in models.items():
    percentages = []
    print(f"\nEvaluating model: {model_name}")
    for file_name in file_names:
        df = pd.read_csv(file_name)
        jokes = df['generated_punchline'].tolist()
        predictions, probabilities = predict_humor_phi(model, phi_tok, jokes)
        n_humorous = list(predictions).count(1)
        percentage_humorous = n_humorous / len(predictions) * 100
        percentages.append(percentage_humorous)
        print(f"File: {file_name}")
        print(f"Number of humorous jokes: {n_humorous}")
        print(f"Percentage of humorous jokes: {percentage_humorous:.2f}%\n")
    mean_percentage = np.mean(percentages)
    std_dev_percentage = np.std(percentages)
    print(f"{model_name} → Mean: {mean_percentage:.2f}%, Std: {std_dev_percentage:.2f}%")